# Network Threat Detection and Risk Prediction

This notebook implements a **two-layer cybersecurity intelligence system** built around two complementary machine-learning models.

### Detective 1: The Threat Spotter

The first model is responsible for **threat detection and classification**.

Given a single network-traffic record, it answers:

> **“What is happening in this traffic?”**

It distinguishes between benign traffic and **34 known attack categories**, learning their characteristic patterns from the **CICIoT2023** dataset.

This model therefore serves as the system's **detection layer**, focusing on identifying and classifying attacks that resemble patterns observed in the training data.

### Detective 2: The Risk Analyst

The second model addresses a different problem.

Instead of asking:

> **“What attack is this?”**

it asks:

> **“How severe does this traffic pattern appear, and does it exhibit characteristics associated with more dangerous activity?”**

This model is trained on the **ToN_IoT** dataset and assigns network activity to predefined **severity/risk tiers**.

The goal is to provide an additional layer of threat intelligence. A traffic sample may not be confidently identified as a particular attack, yet its characteristics may still indicate elevated risk. The risk model is therefore intended to complement, rather than replace, the detection model.

### Important Limitation: No Genuine Temporal Forecasting

A true predictive cybersecurity model would ideally learn from the **temporal progression of attacks**:

```text
Normal Activity
      ↓
Reconnaissance
      ↓
Initial Compromise
      ↓
Privilege Escalation
      ↓
Lateral Movement
      ↓
High-Severity Attack
```

However, the public versions of the CICIoT2023 and ToN_IoT datasets used in this notebook do not provide sufficiently reliable real-world timestamps and event ordering to construct such a temporal sequence.

Consequently, this notebook does **not** claim to perform genuine time-series forecasting.

Instead, the second model performs **severity and risk-tier prediction** based on the characteristics of observed network traffic. This provides a practical approximation of predictive threat assessment while remaining faithful to the information actually available in the datasets.

### Computational Optimization

The original implementation attempted to train deep tree-based models on millions of network-flow records using CPU computation. With approximately **5.5 million records**, this resulted in prohibitively long training times.

This version introduces several optimizations:

* A **capped training sample** is used instead of the entire dataset.
* Sampling is **balanced** to prevent dominant traffic classes from overwhelming minority attack classes.
* **XGBoost** is used for efficient gradient-boosted tree training.
* **GPU acceleration** is enabled to take advantage of Kaggle's NVIDIA T4 hardware.
* Separate training and evaluation datasets are maintained to prevent data leakage.

These changes preserve the core experimental objective while making the notebook computationally practical.

> **Kaggle setup:** Enable the **GPU T4 ×2 accelerator** before running the training cells.

The resulting architecture can therefore be summarized as:

```text
                    Network Traffic
                           │
             ┌─────────────┴─────────────┐
             │                           │
             ▼                           ▼
      Detective 1                  Detective 2
      Threat Spotter               Risk Analyst
             │                           │
             ▼                           ▼
    Attack Classification          Severity Scoring
             │                           │
             │                           │
             └─────────────┬─────────────┘
                           ▼
                 Combined Threat Profile
```

Together, the two models provide complementary perspectives:

**Detection:** *What type of activity is this?*

**Risk assessment:** *How dangerous does this activity appear?*

This separation allows the system to move beyond simple attack classification toward a more useful **threat-intelligence pipeline**, while clearly distinguishing severity prediction from true temporal attack forecasting.


In [1]:
import os, json, time, warnings, shutil, subprocess
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)

import xgboost as xgb
import joblib

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})
sns.set_palette('viridis')

BASE_OUT = '/kaggle/working'
FIG_DIR = os.path.join(BASE_OUT, 'figures')
MODEL_DIR = os.path.join(BASE_OUT, 'models')
REPORT_DIR = os.path.join(BASE_OUT, 'reports')

for d in [FIG_DIR, MODEL_DIR, REPORT_DIR]:
    os.makedirs(d, exist_ok=True)
for sub in ['detection', 'severity']:
    os.makedirs(os.path.join(FIG_DIR, sub), exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def gpu_is_available():
    try:
        result = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=10)
        return result.returncode == 0
    except Exception:
        return False

USE_GPU = gpu_is_available()
DEVICE = 'cuda' if USE_GPU else 'cpu'
print(f"GPU detected: {USE_GPU}. Training device set to: {DEVICE}")
if not USE_GPU:
    print("No GPU found. Go to Notebook Settings > Accelerator > GPU T4 x2, then re-run.")


GPU detected: True. Training device set to: cuda


In [2]:
def savefig(fig, subfolder, name):
    path = os.path.join(FIG_DIR, subfolder, f"{name}.png")
    fig.savefig(path)
    plt.close(fig)
    print(f"Saved chart -> {path}")
    return path


def cap_classes(df, label_col, cap, random_state=RANDOM_STATE):
    """Keep every row for small classes, but trim huge classes down to `cap` rows.
    This is our main speed trick: it shrinks the dataset a lot while actually
    helping class balance instead of hurting it."""
    parts = []
    for cls, grp in df.groupby(label_col):
        if len(grp) > cap:
            grp = grp.sample(cap, random_state=random_state)
        parts.append(grp)
    out = pd.concat(parts).sample(frac=1, random_state=random_state).reset_index(drop=True)
    return out


def plot_class_distribution(series, title, subfolder, name, top_n=40):
    counts = series.value_counts().sort_values(ascending=False)
    if len(counts) > top_n:
        counts = counts.head(top_n)
    fig, ax = plt.subplots(figsize=(12, max(5, len(counts) * 0.3)))
    sns.barplot(x=counts.values, y=counts.index, ax=ax, palette='viridis')
    ax.set_xlabel('Number of Samples')
    ax.set_ylabel('Class')
    ax.set_title(title)
    for i, v in enumerate(counts.values):
        ax.text(v, i, f' {v:,}', va='center', fontsize=8)
    savefig(fig, subfolder, name)


def plot_confusion_matrix(y_true, y_pred, labels, title, subfolder, name, normalize=True, figsize=(12, 10)):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    if normalize:
        cm_display = cm.astype('float') / (cm.sum(axis=1, keepdims=True) + 1e-9)
        fmt = '.2f'
    else:
        cm_display = cm
        fmt = 'd'
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(cm_display, annot=len(labels) <= 20, fmt=fmt, cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax, cbar=True,
                annot_kws={'size': 7})
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.set_title(title)
    plt.setp(ax.get_xticklabels(), rotation=90, ha='center', fontsize=8)
    plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)
    savefig(fig, subfolder, name)
    return cm


def plot_roc_curves_ovr(y_true_bin, y_score, class_names, title, subfolder, name):
    fig, ax = plt.subplots(figsize=(10, 8))
    aucs = []
    n_classes = y_true_bin.shape[1]
    colors = plt.cm.tab20(np.linspace(0, 1, n_classes))
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        if n_classes <= 15:
            ax.plot(fpr, tpr, lw=1.5, color=colors[i], label=f"{class_names[i]} (AUC={roc_auc:.3f})")
        else:
            ax.plot(fpr, tpr, lw=1, color=colors[i], alpha=0.7)
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Chance')
    macro_auc = np.mean(aucs)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f"{title}\nMacro-average AUC = {macro_auc:.4f}")
    if n_classes <= 15:
        ax.legend(loc='lower right', fontsize=7)
    savefig(fig, subfolder, name)
    return dict(zip(class_names, aucs)), macro_auc


def plot_pr_curves_ovr(y_true_bin, y_score, class_names, title, subfolder, name):
    fig, ax = plt.subplots(figsize=(10, 8))
    aps = []
    n_classes = y_true_bin.shape[1]
    colors = plt.cm.tab20(np.linspace(0, 1, n_classes))
    for i in range(n_classes):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_score[:, i])
        ap = average_precision_score(y_true_bin[:, i], y_score[:, i])
        aps.append(ap)
        if n_classes <= 15:
            ax.plot(recall, precision, lw=1.5, color=colors[i], label=f"{class_names[i]} (AP={ap:.3f})")
        else:
            ax.plot(recall, precision, lw=1, color=colors[i], alpha=0.7)
    macro_ap = np.mean(aps)
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f"{title}\nMacro-average AP = {macro_ap:.4f}")
    if n_classes <= 15:
        ax.legend(loc='lower left', fontsize=7)
    savefig(fig, subfolder, name)
    return dict(zip(class_names, aps)), macro_ap


def plot_per_class_metrics(y_true, y_pred, class_names, title, subfolder, name):
    report = classification_report(y_true, y_pred, target_names=class_names,
                                    output_dict=True, zero_division=0)
    df = pd.DataFrame(report).T.iloc[:len(class_names)]
    df = df[['precision', 'recall', 'f1-score']]
    fig, ax = plt.subplots(figsize=(12, max(5, len(class_names) * 0.35)))
    df.plot(kind='barh', ax=ax, width=0.75)
    ax.set_xlabel('Score')
    ax.set_title(title)
    ax.set_xlim(0, 1.05)
    ax.legend(loc='lower right')
    savefig(fig, subfolder, name)
    return df


def plot_feature_importance_from_values(importance_values, feature_names, title, subfolder, name, top_n=30):
    idx = np.argsort(importance_values)[::-1][:top_n]
    fig, ax = plt.subplots(figsize=(10, max(5, top_n * 0.28)))
    sns.barplot(x=np.array(importance_values)[idx], y=np.array(feature_names)[idx], ax=ax, palette='mako')
    ax.set_xlabel('Importance (gain)')
    ax.set_title(title)
    savefig(fig, subfolder, name)


def plot_training_progression(evals_result, title, subfolder, name):
    """This is the training progression chart: loss and error, round by round,
    for both the training set and the validation set side by side."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for split_name, metrics in evals_result.items():
        if 'mlogloss' in metrics:
            axes[0].plot(metrics['mlogloss'], label=split_name)
        if 'merror' in metrics:
            axes[1].plot(metrics['merror'], label=split_name)
    axes[0].set_title('Loss per Boosting Round')
    axes[0].set_xlabel('Round')
    axes[0].set_ylabel('Multiclass Log Loss')
    axes[0].legend()
    axes[1].set_title('Error Rate per Boosting Round')
    axes[1].set_xlabel('Round')
    axes[1].set_ylabel('Multiclass Error Rate')
    axes[1].legend()
    fig.suptitle(title, fontweight='bold')
    savefig(fig, subfolder, name)


def plot_correlation_heatmap(df, title, subfolder, name, max_features=30):
    numeric_df = df.select_dtypes(include=[np.number])
    if numeric_df.shape[1] > max_features:
        variances = numeric_df.var().sort_values(ascending=False)
        numeric_df = numeric_df[variances.head(max_features).index]
    corr = numeric_df.corr()
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax, square=True,
                cbar_kws={'shrink': 0.8}, xticklabels=True, yticklabels=True)
    plt.setp(ax.get_xticklabels(), rotation=90, fontsize=7)
    plt.setp(ax.get_yticklabels(), rotation=0, fontsize=7)
    ax.set_title(title)
    savefig(fig, subfolder, name)


def plot_confidence_histogram(y_score_max, title, subfolder, name):
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.hist(y_score_max, bins=50, color='teal', edgecolor='black', alpha=0.8)
    ax.set_xlabel('Predicted Probability of Chosen Class')
    ax.set_ylabel('Number of Samples')
    ax.set_title(title)
    savefig(fig, subfolder, name)


def plot_metric_summary_bar(metrics_dict, title, subfolder, name):
    fig, ax = plt.subplots(figsize=(8, 5))
    names = list(metrics_dict.keys())
    values = list(metrics_dict.values())
    bars = ax.bar(names, values, color=sns.color_palette('viridis', len(names)))
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.4f}", ha='center', fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title(title)
    savefig(fig, subfolder, name)


def train_xgb_classifier(X_train, y_train, X_val, y_val, num_class, sample_weight=None,
                          max_depth=8, learning_rate=0.15, num_boost_round=250, early_stopping_rounds=20):
    """One training function shared by both detectives. Uses GPU if we found one."""
    dtrain = xgb.DMatrix(X_train, label=y_train, weight=sample_weight, feature_names=list(X_train.columns))
    dval = xgb.DMatrix(X_val, label=y_val, feature_names=list(X_val.columns))

    params = {
        'objective': 'multi:softprob',
        'num_class': num_class,
        'eval_metric': ['mlogloss', 'merror'],
        'tree_method': 'hist',
        'device': DEVICE,
        'max_depth': max_depth,
        'eta': learning_rate,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'min_child_weight': 5,
        'seed': RANDOM_STATE,
    }

    evals_result = {}
    t0 = time.time()
    booster = xgb.train(
        params, dtrain,
        num_boost_round=num_boost_round,
        evals=[(dtrain, 'train'), (dval, 'valid')],
        early_stopping_rounds=early_stopping_rounds,
        evals_result=evals_result,
        verbose_eval=20,
    )
    elapsed = time.time() - t0
    print(f"Done in {elapsed/60:.1f} minutes. Best round: {booster.best_iteration}")
    return booster, evals_result, elapsed


def xgb_predict_proba(booster, X):
    dmat = xgb.DMatrix(X, feature_names=list(X.columns))
    return booster.predict(dmat, iteration_range=(0, booster.best_iteration + 1))


def get_xgb_importance(booster, feature_names, importance_type='gain'):
    score_dict = booster.get_score(importance_type=importance_type)
    return np.array([score_dict.get(f, 0.0) for f in feature_names])

print("Tools ready.")


Tools ready.


---
## Chapter 1: Meet Detective Spotter (Detection Model, CICIoT2023)

Spotter's job is simple to describe and hard to do well: look at one flow of network traffic and say exactly what it is, out of 34 attack types plus normal traffic.

The original CICIoT2023 training file has 5.49 million rows, and a handful of attack types (like the DDoS floods) completely dominate it while a few rare types have only a few hundred examples. Training on all of it with a deep model is what blew our training time past 5 hours last time. So here is the trick: we cap every class at 30,000 rows. Big classes get trimmed down, small classes stay exactly as they are. This alone should shrink the training set from 5.49 million rows to somewhere around 600,000 to 700,000, while actually making the class balance better, not worse.


In [3]:
CICIOT_DIR = '/kaggle/input/datasets/himadri07/ciciot2023/CICIOT23'

train_df = pd.read_csv(os.path.join(CICIOT_DIR, 'train', 'train.csv'))
val_df   = pd.read_csv(os.path.join(CICIOT_DIR, 'validation', 'validation.csv'))
test_df  = pd.read_csv(os.path.join(CICIOT_DIR, 'test', 'test.csv'))

print("Before capping:")
print("Train:", train_df.shape, "| Val:", val_df.shape, "| Test:", test_df.shape)


Before capping:
Train: (5491971, 47) | Val: (1176851, 47) | Test: (1176851, 47)


In [4]:
plot_class_distribution(
    train_df['label'],
    title='CICIoT2023, Full Training Set: Who Is In The Crowd?',
    subfolder='detection', name='01_class_distribution_before_cap'
)

binary_view = train_df['label'].apply(lambda x: 'Benign' if x == 'BenignTraffic' else 'Attack')
fig, ax = plt.subplots(figsize=(6, 6))
binary_view.value_counts().plot.pie(autopct='%1.1f%%', ax=ax, colors=['#4C72B0', '#C44E52'],
                                     startangle=90, textprops={'fontsize': 11})
ax.set_ylabel('')
ax.set_title('CICIoT2023: How Much Of The Traffic Is Actually An Attack?')
savefig(fig, 'detection', '02_benign_vs_attack_pie')

plot_correlation_heatmap(train_df.drop(columns=['label']),
                          title='CICIoT2023: How Do The Features Relate To Each Other?',
                          subfolder='detection', name='03_feature_correlation_heatmap')


Saved chart -> /kaggle/working/figures/detection/01_class_distribution_before_cap.png
Saved chart -> /kaggle/working/figures/detection/02_benign_vs_attack_pie.png
Saved chart -> /kaggle/working/figures/detection/03_feature_correlation_heatmap.png


In [5]:
CAP_PER_CLASS = None

if CAP_PER_CLASS is not None:
    train_df_capped = cap_classes(train_df, 'label', cap=CAP_PER_CLASS)
else:
    train_df_capped = train_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("After capping at", CAP_PER_CLASS, "rows per class:" if CAP_PER_CLASS else "no cap, using full dataset:")
print("Train:", train_df_capped.shape, "(was", train_df.shape, ")")

val_df_sample = val_df.groupby('label', group_keys=False).apply(
    lambda g: g.sample(min(len(g), 6000), random_state=RANDOM_STATE)
)
print("Validation sample used for training checkpoints:", val_df_sample.shape)

plot_class_distribution(
    train_df_capped['label'],
    title=f'CICIoT2023, After Capping at Full Caps Rows Per Class',
    subfolder='detection', name='04_class_distribution_after_cap'
)


After capping at None no cap, using full dataset:
Train: (5491971, 47) (was (5491971, 47) )
Validation sample used for training checkpoints: (125686, 47)
Saved chart -> /kaggle/working/figures/detection/04_class_distribution_after_cap.png


In [6]:
FEATURE_COLS_DET = [c for c in train_df.columns if c != 'label']

le_det = LabelEncoder()
y_train_det = le_det.fit_transform(train_df_capped['label'])
y_val_det   = le_det.transform(val_df_sample['label'])
y_test_det  = le_det.transform(test_df['label'])

X_train_det = train_df_capped[FEATURE_COLS_DET]
X_val_det   = val_df_sample[FEATURE_COLS_DET]
X_test_det  = test_df[FEATURE_COLS_DET]

DET_CLASS_NAMES = list(le_det.classes_)
print(f"Number of classes Spotter must learn: {len(DET_CLASS_NAMES)}")

joblib.dump(le_det, os.path.join(MODEL_DIR, 'detection_label_encoder.joblib'))

classes_arr = np.unique(y_train_det)
class_weights = compute_class_weight('balanced', classes=classes_arr, y=y_train_det)
class_weight_dict = dict(zip(classes_arr, class_weights))
sample_weight_train = np.array([class_weight_dict[c] for c in y_train_det])
print("Class weights ready. Smallest and largest weight:", round(min(class_weights), 3), round(max(class_weights), 3))


Number of classes Spotter must learn: 34
Class weights ready. Smallest and largest weight: 0.19 1153.775


In [7]:
detector_model, evals_result_det, train_time_det = train_xgb_classifier(
    X_train_det, y_train_det, X_val_det, y_val_det,
    num_class=len(DET_CLASS_NAMES),
    sample_weight=sample_weight_train,
    max_depth=8, learning_rate=0.15,
    num_boost_round=250, early_stopping_rounds=20,
)


[0]	train-mlogloss:1.60692	train-merror:0.10487	valid-mlogloss:1.46600	valid-merror:0.05211
[20]	train-mlogloss:0.27398	train-merror:0.04891	valid-mlogloss:0.19441	valid-merror:0.04357
[40]	train-mlogloss:0.15217	train-merror:0.03498	valid-mlogloss:0.12396	valid-merror:0.04078
[60]	train-mlogloss:0.10635	train-merror:0.02697	valid-mlogloss:0.10927	valid-merror:0.03775
[220]	train-mlogloss:0.03016	train-merror:0.00605	valid-mlogloss:0.09128	valid-merror:0.03180
[240]	train-mlogloss:0.02742	train-merror:0.00513	valid-mlogloss:0.09129	valid-merror:0.03167
[249]	train-mlogloss:0.02629	train-merror:0.00478	valid-mlogloss:0.09136	valid-merror:0.03160
Done in 8.8 minutes. Best round: 248


In [8]:
plot_training_progression(evals_result_det,
                           title='Spotter Learning To Tell Attacks Apart, Round By Round',
                           subfolder='detection', name='05_training_progression')


Saved chart -> /kaggle/working/figures/detection/05_training_progression.png


### Report card time

Now that Spotter has finished training, let's see how it does on traffic it has never seen before (the test set, kept completely untouched during training).


In [9]:
y_test_proba_det = xgb_predict_proba(detector_model, X_test_det)
y_test_pred_det = np.argmax(y_test_proba_det, axis=1)

acc_det = accuracy_score(y_test_det, y_test_pred_det)
bal_acc_det = balanced_accuracy_score(y_test_det, y_test_pred_det)
prec_macro_det = precision_score(y_test_det, y_test_pred_det, average='macro', zero_division=0)
rec_macro_det = recall_score(y_test_det, y_test_pred_det, average='macro', zero_division=0)
f1_macro_det = f1_score(y_test_det, y_test_pred_det, average='macro', zero_division=0)
f1_weighted_det = f1_score(y_test_det, y_test_pred_det, average='weighted', zero_division=0)
mcc_det = matthews_corrcoef(y_test_det, y_test_pred_det)
kappa_det = cohen_kappa_score(y_test_det, y_test_pred_det)

print("SPOTTER, TEST SET RESULTS")
print(f"Accuracy:            {acc_det:.4f}")
print(f"Balanced Accuracy:   {bal_acc_det:.4f}")
print(f"Precision (macro):   {prec_macro_det:.4f}")
print(f"Recall (macro):      {rec_macro_det:.4f}")
print(f"F1-score (macro):    {f1_macro_det:.4f}")
print(f"F1-score (weighted): {f1_weighted_det:.4f}")
print(f"Matthews Corr Coef:  {mcc_det:.4f}")
print(f"Cohen's Kappa:       {kappa_det:.4f}")

detection_summary_metrics = {
    'Accuracy': acc_det, 'Balanced Acc.': bal_acc_det, 'Precision': prec_macro_det,
    'Recall': rec_macro_det, 'F1 (macro)': f1_macro_det, 'F1 (weighted)': f1_weighted_det
}
plot_metric_summary_bar(detection_summary_metrics,
                         title='Spotter, Test Set Summary',
                         subfolder='detection', name='06_summary_metrics_bar')


SPOTTER, TEST SET RESULTS
Accuracy:            0.9947
Balanced Accuracy:   0.8663
Precision (macro):   0.9142
Recall (macro):      0.8663
F1-score (macro):    0.8828
F1-score (weighted): 0.9948
Matthews Corr Coef:  0.9942
Cohen's Kappa:       0.9942
Saved chart -> /kaggle/working/figures/detection/06_summary_metrics_bar.png


In [10]:
plot_confusion_matrix(y_test_det, y_test_pred_det, labels=classes_arr,
                       title='Spotter, Normalized Confusion Matrix',
                       subfolder='detection', name='07_confusion_matrix_normalized', normalize=True)

plot_confusion_matrix(y_test_det, y_test_pred_det, labels=classes_arr,
                       title='Spotter, Raw Count Confusion Matrix',
                       subfolder='detection', name='08_confusion_matrix_raw', normalize=False)


Saved chart -> /kaggle/working/figures/detection/07_confusion_matrix_normalized.png
Saved chart -> /kaggle/working/figures/detection/08_confusion_matrix_raw.png


array([[   51,     0,     0, ...,     0,     0,     1],
       [    1, 25459,     9, ...,     0,     0,     1],
       [    0,     6,    89, ...,     0,     0,     0],
       ...,
       [    1,     1,     0, ...,    11,     0,     0],
       [    0,     0,     0, ...,     0,   905,     0],
       [    3,     2,     0, ...,     0,     0,    47]])

# ROC curves, one class against all the rest

In [11]:
y_test_bin_det = label_binarize(y_test_det, classes=classes_arr)
det_auc_per_class, det_macro_auc = plot_roc_curves_ovr(
    y_test_bin_det, y_test_proba_det, DET_CLASS_NAMES,
    title='Spotter, ROC Curves (One Class vs All Others)',
    subfolder='detection', name='09_roc_curves_ovr'
)
print(f"Macro-average ROC-AUC: {det_macro_auc:.4f}")


Saved chart -> /kaggle/working/figures/detection/09_roc_curves_ovr.png
Macro-average ROC-AUC: 0.9998


In [12]:
det_ap_per_class, det_macro_ap = plot_pr_curves_ovr(
    y_test_bin_det, y_test_proba_det, DET_CLASS_NAMES,
    title='Spotter, Precision-Recall Curves (One Class vs All Others)',
    subfolder='detection', name='10_precision_recall_curves_ovr'
)
print(f"Macro-average AP: {det_macro_ap:.4f}")


Saved chart -> /kaggle/working/figures/detection/10_precision_recall_curves_ovr.png
Macro-average AP: 0.8950


In [13]:
per_class_df_det = plot_per_class_metrics(
    y_test_det, y_test_pred_det, DET_CLASS_NAMES,
    title='Spotter, Precision, Recall and F1 for Every Attack Type',
    subfolder='detection', name='11_per_class_metrics'
)
per_class_df_det.to_csv(os.path.join(REPORT_DIR, 'detection_per_class_metrics.csv'))
per_class_df_det


Saved chart -> /kaggle/working/figures/detection/11_per_class_metrics.png


,precision,recall,f1-score
Backdoor_Malware,0.772727,0.573034,0.658065
BenignTraffic,0.959775,0.918799,0.938840
BrowserHijacking,0.741667,0.664179,0.700787
CommandInjection,0.809524,0.571429,0.669951
DDoS-ACK_Fragmentation,0.998493,0.999726,0.999109
DDoS-HTTP_Flood,0.997159,0.990127,0.993631
DDoS-ICMP_Flood,0.999978,0.999956,0.999967
DDoS-ICMP_Fragmentation,0.999123,0.999561,0.999342
DDoS-PSHACK_Flood,1.000000,0.999913,0.999956
DDoS-RSTFINFlood,0.999990,0.999941,0.999966


In [14]:
det_importance = get_xgb_importance(detector_model, FEATURE_COLS_DET)
plot_feature_importance_from_values(det_importance, FEATURE_COLS_DET,
                                     title='Spotter, Top 30 Features It Relies On',
                                     subfolder='detection', name='12_feature_importance')


Saved chart -> /kaggle/working/figures/detection/12_feature_importance.png


In [15]:
max_conf_det = y_test_proba_det.max(axis=1)
plot_confidence_histogram(max_conf_det,
                           title='Spotter, How Confident Are Its Predictions?',
                           subfolder='detection', name='13_confidence_distribution')

correct_mask = (y_test_pred_det == y_test_det)
fig, ax = plt.subplots(figsize=(9, 6))
ax.hist(max_conf_det[correct_mask], bins=40, alpha=0.6, label='Correct calls', color='seagreen')
ax.hist(max_conf_det[~correct_mask], bins=40, alpha=0.6, label='Wrong calls', color='indianred')
ax.set_xlabel('Predicted Probability of Chosen Class')
ax.set_ylabel('Count')
ax.set_title('Spotter, Confidence When Right vs When Wrong')
ax.legend()
savefig(fig, 'detection', '14_confidence_correct_vs_incorrect')


Saved chart -> /kaggle/working/figures/detection/13_confidence_distribution.png
Saved chart -> /kaggle/working/figures/detection/14_confidence_correct_vs_incorrect.png


'/kaggle/working/figures/detection/14_confidence_correct_vs_incorrect.png'

In [16]:
detector_model.save_model(os.path.join(MODEL_DIR, 'detection_model_xgboost.json'))
joblib.dump(detector_model, os.path.join(MODEL_DIR, 'detection_model_xgboost.joblib'))

with open(os.path.join(MODEL_DIR, 'detection_feature_columns.json'), 'w') as f:
    json.dump(FEATURE_COLS_DET, f, indent=2)
with open(os.path.join(MODEL_DIR, 'detection_class_names.json'), 'w') as f:
    json.dump(DET_CLASS_NAMES, f, indent=2)

print("Spotter is saved. Files in the models folder so far:")
for fname in os.listdir(MODEL_DIR):
    print(' -', fname)


Spotter is saved. Files in the models folder so far:
 - detection_label_encoder.joblib
 - detection_class_names.json
 - detection_model_xgboost.joblib
 - detection_feature_columns.json
 - detection_model_xgboost.json


---
## Chapter 2: Meet Detective Risk Analyst (Severity Model, ToN_IoT)

Risk Analyst does not try to name the exact attack. It looks at a flow and decides how serious the situation is, on a five step scale from completely normal up to critical. This is our stand-in for predictive analytics, since we already confirmed that real timestamp based forecasting is not possible with the data we have access to.

**Severity tiers:**
- Tier 0, Normal: nothing to worry about
- Tier 1, Reconnaissance: someone is scanning around, an early warning sign
- Tier 2, Access or Compromise: password attacks, backdoors, man in the middle, injection, cross site scripting
- Tier 3, High: denial of service and distributed denial of service
- Tier 4, Critical: ransomware

This dataset is much smaller (about 211,000 rows), so this half of the notebook should only take a couple of minutes.


In [17]:
TON_PATH = '/kaggle/input/datasets/arnobbhowmik/ton-iot-network-dataset/train_test_network.csv'
ton_df = pd.read_csv(TON_PATH)
print("ToN_IoT shape:", ton_df.shape)
ton_df.head()


ToN_IoT shape: (211043, 44)


,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,-,290.371539,101568,2592,OTH,...,0,0,-,-,-,-,-,-,1,backdoor
1,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000102,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
2,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000148,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
3,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000113,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
4,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000130,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor


In [18]:
plot_class_distribution(ton_df['type'],
                         title='ToN_IoT: What Kinds Of Traffic Are In Here?',
                         subfolder='severity', name='01_class_distribution')

SEVERITY_MAP = {
    'normal': 0,
    'scanning': 1,
    'password': 2, 'backdoor': 2, 'mitm': 2, 'injection': 2, 'xss': 2,
    'dos': 3, 'ddos': 3,
    'ransomware': 4,
}
SEVERITY_NAMES = {0: 'Normal', 1: 'Reconnaissance', 2: 'Access/Compromise', 3: 'High (DoS/DDoS)', 4: 'Critical (Ransomware)'}

ton_df['severity'] = ton_df['type'].map(SEVERITY_MAP)
assert ton_df['severity'].isnull().sum() == 0, "Found an attack type with no severity tier assigned."

plot_class_distribution(ton_df['severity'].map(SEVERITY_NAMES),
                         title='ToN_IoT: How Many Flows Land In Each Severity Tier?',
                         subfolder='severity', name='02_severity_distribution')


Saved chart -> /kaggle/working/figures/severity/01_class_distribution.png
Saved chart -> /kaggle/working/figures/severity/02_severity_distribution.png


In [19]:
DROP_COLS = ['src_ip', 'dst_ip', 'label', 'type', 'severity',
             'dns_query', 'ssl_subject', 'ssl_issuer', 'http_uri',
             'http_user_agent', 'weird_addl']
DROP_COLS = [c for c in DROP_COLS if c in ton_df.columns]

categorical_cols = ton_df.drop(columns=DROP_COLS).select_dtypes(include='object').columns.tolist()
print("Text columns that need encoding:", categorical_cols)

ton_features = ton_df.drop(columns=DROP_COLS).copy()

cat_encoders = {}
for col in categorical_cols:
    enc = LabelEncoder()
    ton_features[col] = enc.fit_transform(ton_features[col].astype(str))
    cat_encoders[col] = enc

FEATURE_COLS_SEV = ton_features.columns.tolist()
X_sev = ton_features
y_sev = ton_df['severity'].values

X_train_sev, X_temp_sev, y_train_sev, y_temp_sev = train_test_split(
    X_sev, y_sev, test_size=0.30, stratify=y_sev, random_state=RANDOM_STATE
)
X_val_sev, X_test_sev, y_val_sev, y_test_sev = train_test_split(
    X_temp_sev, y_temp_sev, test_size=0.50, stratify=y_temp_sev, random_state=RANDOM_STATE
)

print("Train:", X_train_sev.shape, "Val:", X_val_sev.shape, "Test:", X_test_sev.shape)
joblib.dump(cat_encoders, os.path.join(MODEL_DIR, 'severity_categorical_encoders.joblib'))


Text columns that need encoding: ['proto', 'service', 'conn_state', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'http_trans_depth', 'http_method', 'http_version', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_name', 'weird_notice']
Train: (147730, 34) Val: (31656, 34) Test: (31657, 34)


['/kaggle/working/models/severity_categorical_encoders.joblib']

In [20]:
SEV_CLASS_NAMES = [SEVERITY_NAMES[i] for i in sorted(SEVERITY_NAMES.keys())]

sev_classes_arr = np.unique(y_train_sev)
sev_class_weights = compute_class_weight('balanced', classes=sev_classes_arr, y=y_train_sev)
sev_weight_dict = dict(zip(sev_classes_arr, sev_class_weights))
sample_weight_train_sev = np.array([sev_weight_dict[c] for c in y_train_sev])

severity_model, evals_result_sev, train_time_sev = train_xgb_classifier(
    X_train_sev, y_train_sev, X_val_sev, y_val_sev,
    num_class=len(SEV_CLASS_NAMES),
    sample_weight=sample_weight_train_sev,
    max_depth=6, learning_rate=0.15,
    num_boost_round=200, early_stopping_rounds=20,
)


[0]	train-mlogloss:1.27213	train-merror:0.01783	valid-mlogloss:1.27267	valid-merror:0.01921
[20]	train-mlogloss:0.09114	train-merror:0.01136	valid-mlogloss:0.09857	valid-merror:0.01159
[40]	train-mlogloss:0.03157	train-merror:0.00865	valid-mlogloss:0.03797	valid-merror:0.00935
[60]	train-mlogloss:0.02298	train-merror:0.00715	valid-mlogloss:0.02918	valid-merror:0.00828
[80]	train-mlogloss:0.01879	train-merror:0.00612	valid-mlogloss:0.02514	valid-merror:0.00783
[100]	train-mlogloss:0.01634	train-merror:0.00533	valid-mlogloss:0.02306	valid-merror:0.00749
[120]	train-mlogloss:0.01462	train-merror:0.00483	valid-mlogloss:0.02156	valid-merror:0.00717
[140]	train-mlogloss:0.01337	train-merror:0.00441	valid-mlogloss:0.02052	valid-merror:0.00704
[160]	train-mlogloss:0.01242	train-merror:0.00403	valid-mlogloss:0.01996	valid-merror:0.00682
[180]	train-mlogloss:0.01170	train-merror:0.00378	valid-mlogloss:0.01950	valid-merror:0.00670
[189]	train-mlogloss:0.01142	train-merror:0.00374	valid-mlogloss:0

In [21]:
plot_training_progression(evals_result_sev,
                           title='Risk Analyst Learning The Severity Tiers, Round By Round',
                           subfolder='severity', name='03_training_progression')


Saved chart -> /kaggle/working/figures/severity/03_training_progression.png


In [22]:
y_test_proba_sev = xgb_predict_proba(severity_model, X_test_sev)
y_test_pred_sev = np.argmax(y_test_proba_sev, axis=1)

acc_sev = accuracy_score(y_test_sev, y_test_pred_sev)
bal_acc_sev = balanced_accuracy_score(y_test_sev, y_test_pred_sev)
prec_macro_sev = precision_score(y_test_sev, y_test_pred_sev, average='macro', zero_division=0)
rec_macro_sev = recall_score(y_test_sev, y_test_pred_sev, average='macro', zero_division=0)
f1_macro_sev = f1_score(y_test_sev, y_test_pred_sev, average='macro', zero_division=0)
f1_weighted_sev = f1_score(y_test_sev, y_test_pred_sev, average='weighted', zero_division=0)
mcc_sev = matthews_corrcoef(y_test_sev, y_test_pred_sev)
kappa_sev = cohen_kappa_score(y_test_sev, y_test_pred_sev)

print("RISK ANALYST, TEST SET RESULTS")
print(f"Accuracy:            {acc_sev:.4f}")
print(f"Balanced Accuracy:   {bal_acc_sev:.4f}")
print(f"Precision (macro):   {prec_macro_sev:.4f}")
print(f"Recall (macro):      {rec_macro_sev:.4f}")
print(f"F1-score (macro):    {f1_macro_sev:.4f}")
print(f"F1-score (weighted): {f1_weighted_sev:.4f}")
print(f"Matthews Corr Coef:  {mcc_sev:.4f}")
print(f"Cohen's Kappa:       {kappa_sev:.4f}")

severity_summary_metrics = {
    'Accuracy': acc_sev, 'Balanced Acc.': bal_acc_sev, 'Precision': prec_macro_sev,
    'Recall': rec_macro_sev, 'F1 (macro)': f1_macro_sev, 'F1 (weighted)': f1_weighted_sev
}
plot_metric_summary_bar(severity_summary_metrics,
                         title='Risk Analyst, Test Set Summary',
                         subfolder='severity', name='04_summary_metrics_bar')


RISK ANALYST, TEST SET RESULTS
Accuracy:            0.9932
Balanced Accuracy:   0.9937
Precision (macro):   0.9928
Recall (macro):      0.9937
F1-score (macro):    0.9932
F1-score (weighted): 0.9932
Matthews Corr Coef:  0.9909
Cohen's Kappa:       0.9909
Saved chart -> /kaggle/working/figures/severity/04_summary_metrics_bar.png


In [23]:
plot_confusion_matrix(y_test_sev, y_test_pred_sev, labels=sev_classes_arr,
                       title='Risk Analyst, Normalized Confusion Matrix',
                       subfolder='severity', name='05_confusion_matrix_normalized', normalize=True, figsize=(7, 6))

plot_confusion_matrix(y_test_sev, y_test_pred_sev, labels=sev_classes_arr,
                       title='Risk Analyst, Raw Count Confusion Matrix',
                       subfolder='severity', name='06_confusion_matrix_raw', normalize=False, figsize=(7, 6))


Saved chart -> /kaggle/working/figures/severity/05_confusion_matrix_normalized.png
Saved chart -> /kaggle/working/figures/severity/06_confusion_matrix_raw.png


array([[ 7483,     0,     6,    11,     0],
       [    1,  2984,     8,     5,     2],
       [   11,    16, 12083,    46,     1],
       [    8,    30,    69,  5893,     0],
       [    0,     0,     0,     0,  3000]])

In [24]:
y_test_bin_sev = label_binarize(y_test_sev, classes=sev_classes_arr)

sev_auc_per_class, sev_macro_auc = plot_roc_curves_ovr(
    y_test_bin_sev, y_test_proba_sev, SEV_CLASS_NAMES,
    title='Risk Analyst, ROC Curves (One Tier vs All Others)',
    subfolder='severity', name='07_roc_curves_ovr'
)
print(f"Macro-average ROC-AUC: {sev_macro_auc:.4f}")

sev_ap_per_class, sev_macro_ap = plot_pr_curves_ovr(
    y_test_bin_sev, y_test_proba_sev, SEV_CLASS_NAMES,
    title='Risk Analyst, Precision-Recall Curves (One Tier vs All Others)',
    subfolder='severity', name='08_precision_recall_curves_ovr'
)
print(f"Macro-average AP: {sev_macro_ap:.4f}")


Saved chart -> /kaggle/working/figures/severity/07_roc_curves_ovr.png
Macro-average ROC-AUC: 0.9999
Saved chart -> /kaggle/working/figures/severity/08_precision_recall_curves_ovr.png
Macro-average AP: 0.9997


In [25]:
per_class_df_sev = plot_per_class_metrics(
    y_test_sev, y_test_pred_sev, SEV_CLASS_NAMES,
    title='Risk Analyst, Precision, Recall and F1 for Every Severity Tier',
    subfolder='severity', name='09_per_class_metrics'
)
per_class_df_sev.to_csv(os.path.join(REPORT_DIR, 'severity_per_class_metrics.csv'))

sev_importance = get_xgb_importance(severity_model, FEATURE_COLS_SEV)
plot_feature_importance_from_values(sev_importance, FEATURE_COLS_SEV,
                                     title='Risk Analyst, Top Features It Relies On',
                                     subfolder='severity', name='10_feature_importance', top_n=25)
per_class_df_sev


Saved chart -> /kaggle/working/figures/severity/09_per_class_metrics.png
Saved chart -> /kaggle/working/figures/severity/10_feature_importance.png


,precision,recall,f1-score
Normal,0.997334,0.997733,0.997534
Reconnaissance,0.984818,0.994667,0.989718
Access/Compromise,0.993178,0.993913,0.993545
High (DoS/DDoS),0.989589,0.982167,0.985864
Critical (Ransomware),0.999001,1.000000,0.999500


In [26]:
max_conf_sev = y_test_proba_sev.max(axis=1)
plot_confidence_histogram(max_conf_sev,
                           title='Risk Analyst, How Confident Are Its Predictions?',
                           subfolder='severity', name='11_confidence_distribution')


Saved chart -> /kaggle/working/figures/severity/11_confidence_distribution.png


In [27]:
severity_model.save_model(os.path.join(MODEL_DIR, 'severity_model_xgboost.json'))
joblib.dump(severity_model, os.path.join(MODEL_DIR, 'severity_model_xgboost.joblib'))

with open(os.path.join(MODEL_DIR, 'severity_feature_columns.json'), 'w') as f:
    json.dump(FEATURE_COLS_SEV, f, indent=2)
with open(os.path.join(MODEL_DIR, 'severity_class_names.json'), 'w') as f:
    json.dump(SEV_CLASS_NAMES, f, indent=2)
with open(os.path.join(MODEL_DIR, 'severity_tier_mapping.json'), 'w') as f:
    json.dump(SEVERITY_MAP, f, indent=2)

print("Risk Analyst is saved. Files in the models folder now:")
for fname in os.listdir(MODEL_DIR):
    print(' -', fname)


Risk Analyst is saved. Files in the models folder now:
 - severity_tier_mapping.json
 - detection_label_encoder.joblib
 - detection_class_names.json
 - severity_model_xgboost.joblib
 - detection_model_xgboost.joblib
 - detection_feature_columns.json
 - severity_class_names.json
 - detection_model_xgboost.json
 - severity_categorical_encoders.joblib
 - severity_feature_columns.json
 - severity_model_xgboost.json


---
## Chapter 3: Put It All Together And Pack It Up

Both detectives have finished training. Let's line up their report cards side by side, write down some notes for your methodology chapter, and zip everything into one folder you can download straight from Kaggle's Output tab.


In [28]:
results_summary = pd.DataFrame([
    {
        'Model': 'Spotter, Detection (CICIoT2023, 34 classes)',
        'Accuracy': acc_det, 'Balanced Accuracy': bal_acc_det,
        'Precision (macro)': prec_macro_det, 'Recall (macro)': rec_macro_det,
        'F1 (macro)': f1_macro_det, 'F1 (weighted)': f1_weighted_det,
        'ROC-AUC (macro)': det_macro_auc, 'MCC': mcc_det, "Cohen's Kappa": kappa_det,
        'Training Time (minutes)': round(train_time_det / 60, 2)
    },
    {
        'Model': 'Risk Analyst, Severity (ToN_IoT, 5 tiers)',
        'Accuracy': acc_sev, 'Balanced Accuracy': bal_acc_sev,
        'Precision (macro)': prec_macro_sev, 'Recall (macro)': rec_macro_sev,
        'F1 (macro)': f1_macro_sev, 'F1 (weighted)': f1_weighted_sev,
        'ROC-AUC (macro)': sev_macro_auc, 'MCC': mcc_sev, "Cohen's Kappa": kappa_sev,
        'Training Time (minutes)': round(train_time_sev / 60, 2)
    },
])
results_summary.to_csv(os.path.join(REPORT_DIR, 'combined_results_summary.csv'), index=False)
print(results_summary.to_string(index=False))
results_summary


                                      Model  Accuracy  Balanced Accuracy  Precision (macro)  Recall (macro)  F1 (macro)  F1 (weighted)  ROC-AUC (macro)      MCC  Cohen's Kappa  Training Time (minutes)
Spotter, Detection (CICIoT2023, 34 classes)  0.994732           0.866251           0.914236        0.866251    0.882843       0.994832         0.999762 0.994230       0.994229                     8.78
  Risk Analyst, Severity (ToN_IoT, 5 tiers)  0.993240           0.993696           0.992784        0.993696    0.993232       0.993236         0.999911 0.990898       0.990896                     0.04


,Model,Accuracy,Balanced Accuracy,Precision (macro),Recall (macro),F1 (macro),F1 (weighted),ROC-AUC (macro),MCC,Cohen's Kappa,Training Time (minutes)
0,"Spotter, Detection (CICIoT2023, 34 classes)",0.994732,0.866251,0.914236,0.866251,0.882843,0.994832,0.999762,0.994230,0.994229,8.78
1,"Risk Analyst, Severity (ToN_IoT, 5 tiers)",0.993240,0.993696,0.992784,0.993696,0.993232,0.993236,0.999911,0.990898,0.990896,0.04


In [29]:
fig, ax = plt.subplots(figsize=(10, 6))
metrics_to_plot = ['Accuracy', 'Balanced Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1 (macro)', 'ROC-AUC (macro)']
x = np.arange(len(metrics_to_plot))
width = 0.35
ax.bar(x - width/2, results_summary.loc[0, metrics_to_plot], width, label='Spotter (Detection)', color='#4C72B0')
ax.bar(x + width/2, results_summary.loc[1, metrics_to_plot], width, label='Risk Analyst (Severity)', color='#DD8452')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, rotation=20, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Spotter vs Risk Analyst: Side By Side')
ax.legend()
savefig(fig, 'detection', '15_combined_model_comparison')
shutil.copy(os.path.join(FIG_DIR, 'detection', '15_combined_model_comparison.png'),
            os.path.join(FIG_DIR, 'severity', '12_combined_model_comparison.png'))


Saved chart -> /kaggle/working/figures/detection/15_combined_model_comparison.png


'/kaggle/working/figures/severity/12_combined_model_comparison.png'

In [30]:

readme_text = f'''IOT THREAT HUNT: MODEL ARTIFACTS
==================================

SPOTTER, THE DETECTION MODEL (CICIoT2023, 34 attack types + Normal)
---------------------------------------------------------------------
  models/detection_model_xgboost.json      <- load this one in your app
  models/detection_model_xgboost.joblib    <- backup copy
  models/detection_label_encoder.joblib    <- turns a predicted number back into an attack name
  models/detection_feature_columns.json    <- exact column order the model expects
  models/detection_class_names.json        <- the 34 class names, in order

  Loading it in Python (Tkinter app):
      import xgboost as xgb, joblib, json
      booster = xgb.Booster()
      booster.load_model('detection_model_xgboost.json')
      label_encoder = joblib.load('detection_label_encoder.joblib')
      feature_cols = json.load(open('detection_feature_columns.json'))

      dmat = xgb.DMatrix(new_data[feature_cols], feature_names=feature_cols)
      proba = booster.predict(dmat)
      pred_label = label_encoder.inverse_transform(proba.argmax(axis=1))

RISK ANALYST, THE SEVERITY MODEL (ToN_IoT, 5 severity tiers)
----------------------------------------------------------------
  models/severity_model_xgboost.json           <- load this one in your app
  models/severity_model_xgboost.joblib         <- backup copy
  models/severity_categorical_encoders.joblib  <- encoders for the text columns
  models/severity_feature_columns.json         <- exact column order the model expects
  models/severity_class_names.json             <- ['Normal','Reconnaissance','Access/Compromise','High (DoS/DDoS)','Critical (Ransomware)']
  models/severity_tier_mapping.json            <- which raw attack type maps to which tier

REPORTS
-------
  reports/detection_per_class_metrics.csv
  reports/severity_per_class_metrics.csv
  reports/combined_results_summary.csv

FIGURES
-------
  figures/detection/  (15 charts covering class balance, correlation, training progression,
                        confusion matrices, ROC and PR curves, per-class scores, feature
                        importance, confidence checks, and the head-to-head comparison)
  figures/severity/   (12 matching charts for the severity model)

A NOTE FOR THE METHODOLOGY CHAPTER
-----------------------------------
Neither public Kaggle copy of CICIoT2023 or ToN_IoT keeps a real timestamp column, so genuine
time-ordered forecasting was not possible without going back to raw packet captures. This system
implements "predictive analytics" as threat-severity scoring instead: given a flow's own features,
it estimates how advanced or dangerous the pattern looks, which anticipates escalation risk rather
than a specific future moment in time. This is the same interpretation used by many real-world
threat-intelligence platforms when a genuine event timeline is not available.
'''

with open(os.path.join(BASE_OUT, 'README.txt'), 'w') as f:
    f.write(readme_text)

print(readme_text)


IOT THREAT HUNT: MODEL ARTIFACTS

SPOTTER, THE DETECTION MODEL (CICIoT2023, 34 attack types + Normal)
---------------------------------------------------------------------
  models/detection_model_xgboost.json      <- load this one in your app
  models/detection_model_xgboost.joblib    <- backup copy
  models/detection_label_encoder.joblib    <- turns a predicted number back into an attack name
  models/detection_feature_columns.json    <- exact column order the model expects
  models/detection_class_names.json        <- the 34 class names, in order

  Loading it in Python (Tkinter app):
      import xgboost as xgb, joblib, json
      booster = xgb.Booster()
      booster.load_model('detection_model_xgboost.json')
      label_encoder = joblib.load('detection_label_encoder.joblib')
      feature_cols = json.load(open('detection_feature_columns.json'))

      dmat = xgb.DMatrix(new_data[feature_cols], feature_names=feature_cols)
      proba = booster.predict(dmat)
      pred_label = labe

In [31]:
import zipfile

zip_path = os.path.join(BASE_OUT, 'iot_security_artifacts.zip')
if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in [MODEL_DIR, FIG_DIR, REPORT_DIR]:
        for root, _, files in os.walk(folder):
            for file in files:
                full_path = os.path.join(root, file)
                arcname = os.path.relpath(full_path, BASE_OUT)
                zf.write(full_path, arcname)
    zf.write(os.path.join(BASE_OUT, 'README.txt'), 'README.txt')

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"Created {zip_path} ({size_mb:.1f} MB)")
print()
print("Once 'Save & Run All' finishes, open the Output tab on the right side of the")
print("notebook and download 'iot_security_artifacts.zip'. Everything is inside it.")


Created /kaggle/working/iot_security_artifacts.zip (29.9 MB)

Once 'Save & Run All' finishes, open the Output tab on the right side of the
notebook and download 'iot_security_artifacts.zip'. Everything is inside it.


In [35]:
N_PER_CLASS_DETECTION = 60

det_feed_parts = []
for cls_name in DET_CLASS_NAMES:
    subset = test_df[test_df['label'] == cls_name]
    n = min(N_PER_CLASS_DETECTION, len(subset))
    det_feed_parts.append(subset.sample(n, random_state=RANDOM_STATE))
det_feed = pd.concat(det_feed_parts).reset_index(drop=True)
det_feed = det_feed.rename(columns={'label': 'true_label'})

DEVICE_POOL = [
    ('Smart Thermostat', '192.168.1.14'),
    ('Security Webcam', '192.168.1.22'),
    ('Smart TV', '192.168.1.31'),
    ('Wearable Fitness Tracker', '192.168.1.45'),
    ('Smart Door Lock', '192.168.1.52'),
    ('Voice Assistant Speaker', '192.168.1.60'),
    ('Smart Plug', '192.168.1.67'),
    ('IP Camera', '192.168.1.73'),
]
rng = np.random.RandomState(RANDOM_STATE)
devices = rng.choice(len(DEVICE_POOL), size=len(det_feed))
det_feed['device_name'] = [DEVICE_POOL[i][0] for i in devices]
det_feed['device_ip'] = [DEVICE_POOL[i][1] for i in devices]

X_test_sev_full = X_test_sev.copy()
X_test_sev_full['severity'] = y_test_sev

n_rows = len(det_feed)
sev_sampled = X_test_sev_full.sample(n=n_rows, replace=True, random_state=RANDOM_STATE + 1).reset_index(drop=True)

det_feed = det_feed.reset_index(drop=True)
simulation_feed = pd.concat([det_feed, sev_sampled], axis=1)

simulation_feed = simulation_feed.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
simulation_feed.insert(0, 'feed_id', [f"row_{i}" for i in range(len(simulation_feed))])

SIM_DIR = os.path.join(BASE_OUT, 'data')
os.makedirs(SIM_DIR, exist_ok=True)
feed_path = os.path.join(SIM_DIR, 'simulations_feed.csv')
simulation_feed.to_csv(feed_path, index=False)

print("Paired simulation feed built:")
print(f"  Total rows: {len(simulation_feed)}")
print(f"  Detection classes covered: {det_feed['true_label'].nunique()}")
print(f"  No NaNs check, detection columns:", simulation_feed[FEATURE_COLS_DET].isnull().sum().sum())
print(f"  No NaNs check, severity columns:", simulation_feed[FEATURE_COLS_SEV].isnull().sum().sum())
print(f"  Saved to: {feed_path}")

with zipfile.ZipFile(zip_path, 'a', zipfile.ZIP_DEFLATED) as zf:
    zf.write(feed_path, os.path.join('data', 'simulation_feed.csv'))
print(f"Added data/simulation_feed.csv into {zip_path}")

Paired simulation feed built:
  Total rows: 2006
  Detection classes covered: 34
  No NaNs check, detection columns: 0
  No NaNs check, severity columns: 0
  Saved to: /kaggle/working/data/simulations_feed.csv
Added data/simulation_feed.csv into /kaggle/working/iot_security_artifacts.zip
